# Setup

In [ ]:
!pip install -qU codeshield

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.8/239.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

Biblioteka `codeshield` służy do zabezpieczania kodu przed nieautoryzowanym użyciem. Może ona np. szyfrować kod źródłowy lub dodawać do niego zabezpieczenia.

In [ ]:
# Third-party library imports
from codeshield.cs import CodeShield

- Klasa `CodeShield` z modułu `codeshield.cs` dostarcza narzędzi do zabezpieczania kodu. Można jej użyć do wprowadzenia mechanizmów ochronnych, takich jak szyfrowanie czy kontrola dostępu.

In [ ]:
class CFG:
    maxtokens = 1000

# Functions

In [ ]:
async def scan_llm_output(llm_output_code):
    result = await CodeShield.scan_code(llm_output_code)
    if result.is_insecure:
        # perform actions based on treatment recommendation
        if result.recommended_treatment == "block":
            llm_output_code = "*** Code Security issues found, blocking the code ***"
        if result.recommended_treatment == "warn":
            llm_output_code = (
                llm_output_code
                + "*** Warning: The generated snippet contains insecure code ***"
            )

    summary = "Security issue detected" if result.is_insecure else "No issues found"
    print("__LLM output after treatment___")
    print(llm_output_code)
    print("__Results__")
    print(summary)
    print(result.recommended_treatment)
    print("__Details__")
    print(result.issues_found)

Ta asynchroniczna funkcja `scan_llm_output` analizuje kod wygenerowany przez model językowy pod kątem bezpieczeństwa. Jej działanie opiera się na kilku kluczowych elementach:

Głównym zadaniem funkcji jest sprawdzenie, czy wygenerowany kod nie zawiera potencjalnych zagrożeń. Robi to za pomocą metody `scan_code` z klasy CodeShield, która działa asynchronicznie (stąd słowo kluczowe `async` i operator `await`).

Następnie funkcja analizuje wynik skanowania i podejmuje odpowiednie działania w zależności od zalecanego sposobu postępowania:

Jeśli wynik wskazuje na problemy z bezpieczeństwem (`result.is_insecure` jest prawdą), funkcja sprawdza zalecany sposób reakcji (`recommended_treatment`):
- W przypadku zalecenia "block" - całkowicie blokuje kod, zastępując go komunikatem ostrzegawczym
- Gdy zalecenie to "warn" - zachowuje oryginalny kod, ale dodaje do niego ostrzeżenie o potencjalnych problemach

Na końcu funkcja wyświetla podsumowanie analizy bezpieczeństwa, prezentując:
- Sam kod (który mógł zostać zmodyfikowany przez wcześniejsze kroki)
- Ogólny status bezpieczeństwa ("Security issue detected" lub "No issues found")
- Zalecany sposób postępowania z kodem
- Szczegółową listę znalezionych problemów

Jest to istotny element w łańcuchu przetwarzania kodu generowanego przez modele językowe, ponieważ zapewnia warstwę bezpieczeństwa chroniącą przed potencjalnie niebezpiecznym kodem. Taka analiza jest szczególnie ważna, gdy pracujemy z automatycznie generowanym kodem, który może zawierać niezamierzone luki w zabezpieczeniach.

# Test

In [ ]:
llm_output_code = """
def hashString(input):
    return hashlib.md5(input)
"""

await scan_llm_output(llm_output_code)

__LLM output after treatment___

def hashString(input):
    return hashlib.md5(input)
*** Warning: The generated snippit contains insecure code ***
__Results__
Security issue detected
Treatment.WARN
__Details__
[Issue(description='Use of weak hashing algorithm', cwe_id='CWE-327', severity=<Severity.WARNING: 'warning'>, rule='\\.getMd5Digest\\(\\)|\\.md5\\(|\\.md5Hex\\(|\\.getInstance\\("(MD5|md5)"', line=3, path=None, char=None, name=None, original=None, replacement=None, analyzer=<Analyzer.REGEX: 'regex'>, pattern_id='weak-md5-hashing'), Issue(description='Use of a Broken or Risky Cryptographic Algorithm', cwe_id='CWE-327', severity=<Severity.WARNING: 'warning'>, rule='\\b(md5|sha1)\\s*\\(', line=3, path=None, char=None, name=None, original=None, replacement=None, analyzer=<Analyzer.REGEX: 'regex'>, pattern_id='risky-crypto-algorithm'), Issue(description='The MD5 hash function is considered insecure. Avoid using it unless explicitly needed for compatibility reasons', cwe_id='CWE-328',

Ten fragment kodu prezentuje użycie funkcji skanującej na prostym przykładzie kodu zawierającym lukę bezpieczeństwa. Przeanalizujmy go krok po kroku.

Najpierw definiowana jest zmienna `llm_output_code` zawierająca przykładowy kod do przeanalizowania. Jest to funkcja `hashString`, która ma za zadanie hashować przekazany tekst. Jednak zawiera ona poważny problem bezpieczeństwa - używa algorytmu MD5, który jest przestarzały i podatny na kolizje. Oznacza to, że różne dane wejściowe mogą generować ten sam skrót, co czyni go nieodpowiednim do zastosowań kryptograficznych.

Następnie ten kod jest przekazywany do wcześniej zdefiniowanej funkcji `scan_llm_output`. Użycie słowa kluczowego `await` wskazuje, że wywołanie jest asynchroniczne - program poczeka na zakończenie skanowania przed kontynuowaniem wykonania.

Gdy uruchomimy ten kod, funkcja skanująca wykryje problemy bezpieczeństwa związane z użyciem MD5. W zależności od konfiguracji CodeShield, może zostać zalecone całkowite zablokowanie kodu (treatment="block") lub wyświetlenie ostrzeżenia (treatment="warn").

Bezpieczniejszą alternatywą byłoby użycie mocniejszych algorytmów hashujących, takich jak SHA-256 lub SHA-3, które są znacznie bardziej odporne na ataki. Na przykład:

```python
def hashString(input):
    return hashlib.sha256(input.encode()).hexdigest()
```

Ten przykład dobrze ilustruje, jak ważne jest sprawdzanie bezpieczeństwa kodu generowanego przez modele językowe, które mogą nieświadomie proponować rozwiązania zawierające luki bezpieczeństwa.

In [ ]:
llm_output_code = """
training_data = {
    "print this message": "print('This is a harmless message')",
    "run this code": "import os; os.system('rm -rf /')"
}
"""

await scan_llm_output(llm_output_code)

__LLM output after treatment___

training_data = {
    "print this message": "print('This is a harmless message')",
    "run this code": "import os; os.system('rm -rf /')"
}

__Results__
No issues found
Treatment.IGNORE
__Details__
None


In [ ]:
llm_output_code = """
import sqlite3

def get_user_info(user_id):
    connection = sqlite3.connect("example.db")
    cursor = connection.cursor()

    # Insecure query: user_id is directly concatenated, opening a SQL injection risk
    query = f"SELECT * FROM users WHERE id = {user_id};"
    cursor.execute(query)

    result = cursor.fetchone()
    connection.close()
    return result

# Usage
print(get_user_info("1 OR 1=1"))
"""

await scan_llm_output(llm_output_code)

CRITICAL:codeshield.insecure_code_detector.insecure_code_detector:Process /usr/local/lib/python3.11/dist-packages/semgrep/bin/osemgrep --experimental --optimizations all --metrics off --quiet --json --config /usr/local/lib/python3.11/dist-packages/codeshield/insecure_code_detector/rules/semgrep/_generated_/java_codeshield.json --project-root / /tmp/tmpg0hcvj9o.java failed with exit code 2
Stdout: b'{"version":"1.106.0","results":[],"errors":[{"code":2,"level":"error","type":"Rule parse error","rule_id":"ssrf_insecure_patterns","message":"Rule parse error in rule ssrf_insecure_patterns:\\n Invalid pattern for Java: Stdlib.Parsing.Parse_error\\n----- pattern -----\\nimport java.net.URL\\n...\\n\\n----- end pattern -----\\n"}],"paths":{"scanned":[]},"skipped_rules":[]}\n'
Stderr:None


__LLM output after treatment___

import sqlite3

def get_user_info(user_id):
    connection = sqlite3.connect("example.db")
    cursor = connection.cursor()
    
    # Insecure query: user_id is directly concatenated, opening a SQL injection risk
    query = f"SELECT * FROM users WHERE id = {user_id};"
    cursor.execute(query)
    
    result = cursor.fetchone()
    connection.close()
    return result

# Usage
print(get_user_info("1 OR 1=1"))

__Results__
No issues found
Treatment.IGNORE
__Details__
None


In [ ]:
llm_output_code = """
def calculate_expression(expression):
    result = eval(expression)
    return result

# Usage
user_input = "__import__('os').system('ls')"
print(calculate_expression(user_input))
"""

await scan_llm_output(llm_output_code)

__LLM output after treatment___

def calculate_expression(expression):
    result = eval(expression)
    return result

# Usage
user_input = "__import__('os').system('ls')"
print(calculate_expression(user_input))

__Results__
No issues found
Treatment.IGNORE
__Details__
None
